# Classifying heart sounds with spectrograms
## A guided comparison of a CNN, an LSTM, and transfer learning

### The question
Can we use a short recording of a heartbeat to predict whether it is **normal** or **abnormal**?

We will work with phonocardiograms (PCGs): audio signals recorded with a digital stethoscope. The notebook turns each sound into a time--frequency image (a **log-mel spectrogram**) and compares three neural-network approaches.

| Model | How it reads the same spectrogram |
|---|---|
| **CNN** | Like an image: it learns local time--frequency patterns. |
| **LSTM** | Like a sequence: it reads one time frame after another. |


### The workflow

`heart-sound recording → filter → 3-second windows → log-mel spectrogram → grouped split → train models → patient-level evaluation`

### Before you run

The PhysioNet/CinC Challenge 2016 validation data must be available in the `heart_sounds/` folder. The first code cell imports the required packages; install them first if needed:

`pip install tensorflow librosa scikit-learn matplotlib pandas`

## Step 0 - Set up the Python environment

Import the libraries used for audio processing, plotting, splitting the data, evaluating models, and building neural networks and fix random seed


In [ ]:
import io, os, glob, zipfile, urllib.request, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"     # hide TensorFlow info messages

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt
from IPython.display import Audio, display

from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay,
                             classification_report)

import keras
from keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)

## 1. Download and load the recordings
The zip is downloaded once and extracted to  `heart_sounds/` folder.

## Step 1 - Load the recordings and their labels

`REFERENCE.csv` reads the recording names and labels; load every `.wav` signal into memory. Labels are converted to `0 = normal` and `1 = abnormal`. Each waveform is scaled to the same amplitude range.

Scaling prevents the model from using recording volume alone. 

In [ ]:
# DATA_URL = "https://physionet.org/files/challenge-2016/1.0.0/validation.zip"
DATA_DIR = "heart_sounds"

ref_path = glob.glob(os.path.join(DATA_DIR, "**", "REFERENCE.csv"), recursive=True)[0]
wav_dir = os.path.dirname(ref_path)

ref = pd.read_csv(ref_path, header=None, names=["record", "label"])
ref["label"] = (ref["label"] == 1).astype(int)     # 1 = abnormal, 0 = normal
ref["database"] = ref["record"].str[0]             # a, b, c, d, e = source database

signals, durations = {}, []
for rec in ref["record"]:
    sr, x = wavfile.read(os.path.join(wav_dir, rec + ".wav"))
    x = x.astype("float32")
    signals[rec] = x / (np.abs(x).max() + 1e-9)    # scale amplitude to [-1, 1]
    durations.append(len(x) / sr)
ref["duration_s"] = durations
SR = sr
print(f"{len(ref)} recordings, sampling rate = {SR} Hz")
ref.head()

### Step 1.1 - Inspect the dataset

Plot the number of recordings in each class, and their durations.


### 1.1 Quick exploration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 3.8))
ref["label"].map({0: "Normal", 1: "Abnormal"}).value_counts().plot.bar(ax=axes[0], color=["#4C72B0", "#C44E52"], rot=0)
axes[0].set_title("Recordings per class")

# pd.crosstab(ref["database"], ref["label"].map({0: "Normal", 1: "Abnormal"})).plot.bar(ax=axes[1], rot=0, color=["#C44E52", "#4C72B0"])
# axes[1].set_title("Recordings per source database")

for lab, name, c in [(0, "Normal", "#4C72B0"), (1, "Abnormal", "#C44E52")]:
    axes[1].hist(ref.loc[ref.label == lab, "duration_s"], bins=25, alpha=0.6, label=name, color=c)
axes[1].set_xlabel("duration (s)"); axes[1].set_title("Recording length"); axes[1].legend()
plt.tight_layout(); plt.show()
ref.groupby("label")["duration_s"].describe().round(1)

Recordings have **different lengths** (from a few seconds to over a minute). Neural networks need inputs of a fixed size → in section 3 we will cut every recording into windows of equal length.

## 2. Listen and look

### 2.1 Listen to one example from each class

**What this code does:** plays the first 10 seconds of one normal and one abnormal recording.

**Why it matters:** machine learning should stay connected to the real signal. Listening builds intuition for what the model is asked to distinguish, even though clinical interpretation requires expertise.


In [ ]:
import sounddevice as sd

ex_normal = ref.loc[ref.label == 0, "record"].iloc[0]
ex_abnormal = ref.loc[ref.label == 1, "record"].iloc[0]

for name, rec in [("Normal", ex_normal), ("Abnormal", ex_abnormal)]:
    print(f"Playing {name}: {rec}")
    sd.play(signals[rec][: SR * 5], SR)
    sd.wait()   # blocks until playback finishes

### 2.2 From sound to image
- **Waveform:** amplitude over time.
- **Spectrogram (STFT):** the signal is cut into short overlapping frames, and for each frame we compute *which frequencies* are present (Fourier transform). Result: a 2D image, x = time, y = frequency, colour = energy.
- **Mel-spectrogram:** the frequency axis is compressed into *mel bands* (finer resolution at low frequencies), and energy is shown in **decibels** (log scale). Fewer rows → smaller images → faster training.

In [ ]:
N_FFT, HOP, N_MELS, FMAX = 256, 64, 40, 800       # 128 ms window, 32 ms hop at 2 kHz

fig, axes = plt.subplots(3, 2, figsize=(15, 9))
for col, (name, rec) in enumerate([("Normal", ex_normal), ("Abnormal", ex_abnormal)]):
    x = signals[rec][: SR * 5]                     # first 5 seconds
    axes[0, col].plot(np.arange(len(x)) / SR, x, lw=0.6)
    axes[0, col].set_title(f"{name} ({rec}) — waveform"); axes[0, col].set_xlabel("time (s)")

    S = librosa.amplitude_to_db(np.abs(librosa.stft(x, n_fft=N_FFT, hop_length=HOP)), ref=np.max)
    librosa.display.specshow(S, sr=SR, hop_length=HOP, x_axis="time", y_axis="hz", ax=axes[1, col])
    axes[1, col].set_title("Spectrogram (STFT, dB)")

    M = librosa.power_to_db(librosa.feature.melspectrogram(y=x, sr=SR, n_fft=N_FFT, hop_length=HOP,
                                                           n_mels=N_MELS, fmax=FMAX), ref=np.max)
    librosa.display.specshow(M, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", fmax=FMAX, ax=axes[2, col])
    axes[2, col].set_title(f"Log-mel spectrogram ({N_MELS} mel bands)")
plt.tight_layout(); plt.show()

Heart sounds live mainly below ~400 Hz; murmurs spread energy to higher frequencies.

## 3. Pre-processing

### 3.1 Keep the frequency range of interest

Define and apply a 25–400 Hz band-pass filter.

Most clinically useful heart-sound energy is in this range. Filtering reduces slow movement/breathing drift and high-frequency handling noise, so later features focus on the information of interest.


In [ ]:
def bandpass(x, low=25, high=400, sr=SR, order=4):
    sos = butter(order, [low, high], btype="bandpass", fs=sr, output="sos")
    return sosfiltfilt(sos, x).astype("float32")

filtered = {rec: bandpass(x) for rec, x in signals.items()}

x_raw, x_f = signals[ex_abnormal][: SR * 3], filtered[ex_abnormal][: SR * 3]
t = np.arange(len(x_raw)) / SR
plt.figure(figsize=(14, 3))
plt.plot(t, x_raw, lw=0.6, alpha=0.6, label="raw")
plt.plot(t, x_f, lw=0.6, label="band-pass 25–400 Hz")
plt.xlabel("time (s)"); plt.legend(); plt.title("Effect of the band-pass filter"); plt.show()

### 3.2 Cut into fixed-length windows
Each recording is cut into **3-second windows** (≈ 3–4 heartbeats) with 50% overlap. Every window inherits the label of its recording.
We also keep the **recording ID** of every window: we will need it for the split.

In [ ]:
WIN_S, STEP_S = 3.0, 1.5
WIN, STEP = int(WIN_S * SR), int(STEP_S * SR)

segments, seg_labels, seg_records = [], [], []
for _, row in ref.iterrows():
    x = filtered[row.record]
    for start in range(0, len(x) - WIN + 1, STEP):
        segments.append(x[start:start + WIN])
        seg_labels.append(row.label)
        seg_records.append(row.record)

segments = np.array(segments)
y = np.array(seg_labels)
groups = np.array(seg_records)
print(f"{len(segments)} windows from {len(ref)} recordings  (window = {WIN} samples)")
print("Windows per class:", dict(zip(*np.unique(y, return_counts=True))))
print("Windows per recording: min", pd.Series(groups).value_counts().min(),
      "| median", int(pd.Series(groups).value_counts().median()),
      "| max", pd.Series(groups).value_counts().max())

Long recordings produce many windows, short ones only a few - one patient can contribute dozens of nearly identical windows.

## 4. Feature extraction: log-mel spectrograms
Every 3-second window becomes a **40 × 94 image** (40 mel bands × 94 time frames).

In [ ]:
def logmel(x):
    M = librosa.feature.melspectrogram(y=x, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, fmax=FMAX)
    return librosa.power_to_db(M, ref=1.0).astype("float32")

start = time.time()
X = np.stack([logmel(s) for s in segments])          # (n_windows, n_mels, n_frames)
print("Spectrogram dataset:", X.shape, f"— computed in {time.time() - start:.1f} s")

fig, axes = plt.subplots(2, 6, figsize=(16, 4.5), sharex=True, sharey=True)
rng = np.random.default_rng(SEED)
for row, lab in enumerate([0, 1]):
    for ax, i in zip(axes[row], rng.choice(np.where(y == lab)[0], 6, replace=False)):
        ax.imshow(X[i], origin="lower", aspect="auto", cmap="magma")
        ax.set_title(groups[i], fontsize=9)
    axes[row, 0].set_ylabel(["Normal", "Abnormal"][lab] + "\nmel band")
plt.suptitle("Random log-mel windows")
plt.tight_layout(); plt.show()

## 5. Train / validation / test split - by recording!

**Classic mistake:** splitting the *windows* at random. Windows of the same recording are almost identical (same patient, same stethoscope, overlapping audio).
If some go to training and some to test, the model can recognise the *patient* instead of the *disease* → over-optimistic results (**data leakage**).

**Solution:** `GroupShuffleSplit` with `groups = recording ID` → all windows of one recording end up on the same side.
We split twice: first 70% train / 30% rest, then the rest into 15% validation / 15% test.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, rest_idx = next(gss.split(X, y, groups))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_rel, test_rel = next(gss2.split(X[rest_idx], y[rest_idx], groups[rest_idx]))
val_idx, test_idx = rest_idx[val_rel], rest_idx[test_rel]

for name, idx in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    print(f"{name:10s}: {len(idx):5d} windows from {len(np.unique(groups[idx])):3d} recordings, "
          f"abnormal = {y[idx].mean():.0%}")

# Sanity check: no recording appears in two sets
assert not set(groups[train_idx]) & set(groups[test_idx])
assert not set(groups[train_idx]) & set(groups[val_idx])
assert not set(groups[val_idx]) & set(groups[test_idx])
print("\nNo recording is shared between train, validation and test ✔")

## 6. Normalisation
Neural networks train better when inputs are around 0 with similar scale. We standardise every mel band using the **mean and standard deviation of the training set only** (as with tabular data).

In [ ]:
mu = X[train_idx].mean(axis=(0, 2), keepdims=True)    # one mean per mel band
sd = X[train_idx].std(axis=(0, 2), keepdims=True) + 1e-6
Xn = (X - mu) / sd

X_train, y_train = Xn[train_idx], y[train_idx]
X_val, y_val = Xn[val_idx], y[val_idx]
X_test, y_test = Xn[test_idx], y[test_idx]

# Class weights: compensate if one class has more windows
w = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight = {0: float(w[0]), 1: float(w[1])}
print("Class weights:", {k: round(v, 2) for k, v in class_weight.items()})

N_MELS_, N_FRAMES = X_train.shape[1], X_train.shape[2]

## 7. Model 1 - 2D Convolutional Neural Network

Input shape for a CNN: **(height, width, channels)** = (40 mel bands, 94 frames, 1).

- `Conv2D(16, (3, 3))`: 16 filters of 3×3 pixels slide over the spectrogram and learn local patterns
- `BatchNormalization`: keeps activations on a stable scale → faster, more stable training
- `MaxPooling2D(2)`: halves height and width
- `GlobalAveragePooling2D`: one number per filter
- `Dense(1, sigmoid)`: probability that the window is abnormal

In [ ]:
def build_cnn():
    model = keras.Sequential([
        keras.Input(shape=(N_MELS_, N_FRAMES, 1)),
        layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid"),
    ], name="CNN")
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
                  metrics=["accuracy", keras.metrics.AUC(name="auc")])
    return model

cnn = build_cnn()
cnn.summary()

## 8. Model 2 - LSTM

For the LSTM the same spectrogram is read as a **sequence over time**: 94 time steps, each a vector of 40 mel energies.
We just transpose the image: (mel, frames) → (frames, mel).

`Bidirectional(LSTM(32))` reads the sequence forwards and backwards (S1 → S2 and S2 → S1) and concatenates the two final memories.

In [ ]:
def to_sequence(X):            # (n, mel, frames) -> (n, frames, mel)
    return np.transpose(X, (0, 2, 1))

def build_lstm():
    model = keras.Sequential([
        keras.Input(shape=(N_FRAMES, N_MELS_)),
        layers.Bidirectional(layers.LSTM(32)),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid"),
    ], name="LSTM")
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
                  metrics=["accuracy", keras.metrics.AUC(name="auc")])
    return model

lstm = build_lstm()
lstm.summary()

## 9. Training the CNN and the LSTM
Early stopping monitors the **validation loss** and keeps the weights of the best epoch.
The cell prepares the correctly shaped input for each model, trains both with the same training/validation split, applies class weights, and stops when validation loss has not improved for several epochs.

In [ ]:
EPOCHS, BATCH_SIZE = 40, 32

inputs = {
    "CNN":  (X_train[..., None],     X_val[..., None],     X_test[..., None]),
    "LSTM": (to_sequence(X_train),   to_sequence(X_val),   to_sequence(X_test)),
}
models = {"CNN": cnn, "LSTM": lstm}
histories, train_times = {}, {}

for name, model in models.items():
    Xtr, Xva, _ = inputs[name]
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    t0 = time.time()
    h = model.fit(Xtr, y_train, validation_data=(Xva, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE,
                  class_weight=class_weight, callbacks=[stop], verbose=2)
    train_times[name] = time.time() - t0
    histories[name] = pd.DataFrame(h.history)
    print(f"--> {name}: {len(h.history['loss'])} epochs, {train_times[name]:.0f} s\n")

### 9.1 Read the learning curves

Plot training and validation loss and AUC after every epoch.

The validation curves estimate how well the model generalises. Training performance improving while validation performance worsens is a warning sign of overfitting.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex="col")
for col, (name, h) in enumerate(histories.items()):
    axes[0, col].plot(h["loss"], label="train"); axes[0, col].plot(h["val_loss"], label="validation")
    axes[0, col].set_title(f"{name} — loss"); axes[0, col].legend()
    axes[1, col].plot(h["auc"], label="train"); axes[1, col].plot(h["val_auc"], label="validation")
    axes[1, col].set_title(f"{name} — AUC"); axes[1, col].set_xlabel("epoch"); axes[1, col].legend()
plt.tight_layout(); plt.show()

### 9.2 Evaluate each 3-second window

**What this code does:** generates an abnormality probability for every test window and prints ROC-AUC plus a classification report using a 0.5 threshold.

**Why it matters:** window-level metrics show how reliably a model handles short excerpts. ROC-AUC evaluates ranking across all possible thresholds; the classification report depends on the chosen threshold.


In [ ]:
win_proba = {
    name: model.predict(inputs[name][2], verbose=0).ravel()
    for name, model in models.items()
}

for name, p in win_proba.items():
    print(f"=== {name} — per window (threshold 0.5), ROC-AUC = {roc_auc_score(y_test, p):.3f} ===")
    print(classification_report(y_test, (p >= 0.5).astype(int), target_names=["Normal", "Abnormal"], digits=3, zero_division=0))

### 9.3 Recording (patient) level
In the clinic we diagnose a **patient recording**, not a 3-second window.
We average the window probabilities of each recording → one probability per recording. This usually improves performance because random errors on single windows cancel out.

In [ ]:
test_groups = groups[test_idx]
rec_results = {}
for name, p in win_proba.items():
    df_ = pd.DataFrame({"record": test_groups, "proba": p, "label": y_test})
    rec_results[name] = df_.groupby("record").agg(proba=("proba", "mean"), label=("label", "first"))

summary = pd.DataFrame([{
    "model": name,
    "parameters": models[name].count_params(),
    "training time (s)": round(train_times[name]),
    "AUC per window": round(roc_auc_score(y_test, win_proba[name]), 3),
    "AUC per recording": round(roc_auc_score(r["label"], r["proba"]), 3),
    "n test recordings": len(r),
} for name, r in rec_results.items()]).set_index("model")
summary

### Plot discrimination and errors

**What this code does:** draws ROC curves at both window and recording level, then confusion matrices for recording-level predictions.

**Why it matters:** ROC curves compare threshold-independent discrimination; confusion matrices reveal the practical consequences at the selected 0.5 threshold—false negatives and false positives.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for name in models:
    RocCurveDisplay.from_predictions(y_test, win_proba[name], name=name, ax=axes[0])
    r = rec_results[name]
    RocCurveDisplay.from_predictions(r["label"], r["proba"], name=name, ax=axes[1])
for ax, title in zip(axes, ["Per window", "Per recording (mean of windows)"]):
    ax.plot([0, 1], [0, 1], "k--", label="chance")
    ax.set_title(f"ROC - {title}")
    ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(rec_results), figsize=(5.5 * len(rec_results), 4.5))
for ax, (name, r) in zip(axes, rec_results.items()):
    ConfusionMatrixDisplay.from_predictions(r["label"], (r["proba"] >= 0.5).astype(int),
                                            display_labels=["Normal", "Abnormal"], cmap="Blues",
                                            colorbar=False, ax=ax)
    tn, fp, fn, tp = pd.crosstab(r["label"], (r["proba"] >= 0.5).astype(int)).reindex(
        index=[0, 1], columns=[0, 1], fill_value=0).values.ravel()
    ax.set_title(f"{name} - per recording\nsensitivity = {tp/(tp+fn):.2f}, specificity = {tn/(tn+fp):.2f}")
plt.tight_layout(); plt.show()

## 10. Exercises
1. **Leakage experiment:** replace `GroupShuffleSplit` with a random `train_test_split` on the windows. What happens to the test AUC? Why is it misleading?
2. Change the window length `WIN_S` to 1.5 s and 5 s. More windows vs more context — which works better?
3. Remove the band-pass filter. Does performance change?
4. Use the plain STFT spectrogram instead of the mel-spectrogram (hint: `np.abs(librosa.stft(...))` → much taller images).
5. CNN: add a third `MaxPooling2D` or more filters. Does the model overfit more?
7. LSTM: remove `Bidirectional`, or stack two LSTMs (`return_sequences=True` in the first).
8. **CNN + LSTM:** use a few `Conv2D` + `MaxPooling2D((2, 1))` layers to extract features, reshape to (time, features) and feed an LSTM.
9. Aggregate windows with the **median** or **max** probability instead of the mean.
10. More data: download `training.zip` (~170 MB, 3,153 recordings) from the same PhysioNet page and repeat the analysis. Note the class imbalance there!

*Data reference:* Clifford G.D. et al., *Classification of Normal/Abnormal Heart Sound Recordings: the PhysioNet/Computing in Cardiology Challenge 2016*, Computing in Cardiology 2016; Goldberger A. et al., PhysioBank, PhysioToolkit and PhysioNet, Circulation 2000.